# Suite VLG — The Verilog language

Verilog and SystemVerilog as one tree language (`@mbse/programs/Verilog/Syntax`): its kinds and categories,
organized as IEEE 1800's grammar is, the identifiers it accepts, and where each kind and feature exists in the two
families, which the standards check.

In [ ]:
import { SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard } from "@mbse/programs/Verilog";

import { assert, equal } from "./support.js";

const L = S.LANGUAGE;
const [V1995, V2001] = [new VerilogStandard(1995, "Verilog"), new VerilogStandard(2001, "Verilog")];
const [SV2005, SV2009, SV2012] = [new VerilogStandard(2005), new VerilogStandard(2009), new VerilogStandard(2012)];
const name = (spelling: unknown) => new S.Identifier({ spelling });
const ref = (spelling: string) => new S.NameExpression({ name: name(spelling) });
const number = (spelling = "1") => new S.IntegerLiteral({ spelling });
const unit = (...items: any[]) => new S.SourceText({ items });
const module = (items: any[] = [], properties: object = {}) =>
  new S.ModuleDeclaration({ keyword: "module", name: name("m"), items, ...properties });

## VLG-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
const kinds = L.kinds();
assert(kinds.size === S.KINDS.length && S.KINDS.length === 117);
equal([...kinds.keys()], S.KINDS.map((k) => k.name));
equal([...L.categories().keys()].sort(), ["Name", "Item", "Port", "DataType", "Dimension", "Statement", "Connection",
  "TimingControl", "Expression", "Literal", "Range", "Directive"].sort());
assert(new S.IntegerLiteral() instanceof S.Literal && new S.IntegerLiteral() instanceof S.Expression);
assert(S.KINDS.every((k) => k.NAME === `Programs.Verilog.${k.KIND}`));
const grammar = L.grammar() as any;
equal(grammar.base, { Verilog: 1995, SystemVerilog: 2005 });
assert(grammar.kinds.length === 117);
equal([S.sv(), S.sv(2009), S.verilog(2001)], [{ SystemVerilog: 2005 }, { SystemVerilog: 2009 },
  { Verilog: 2001, SystemVerilog: 2005 }]);
assert(new S.Comment({ text: " x" }).block === false && new S.SourceText().items.length === 0);
console.log("ok");

## VLG-02 · Identifiers are simple or escaped, and system names start with `$`

In [ ]:
equal(L.validate(name("x_1$")), []);
equal(L.validate(name("\\bus+index")), []);
equal(L.validate(name("")), []);
for (const bad of ["1x", "$x", "a b", "Größe", "\\", "\\a b"]) {
  equal(L.validate(name(bad)), [`${bad === "\\" ? "'\\\\'" : bad === "\\a b" ? "'\\\\a b'" : `'${bad}'`} is not an identifier`]);
}
equal(L.validate(name(3n)), ["Identifier.spelling must be a str, got int"]);
equal(L.validate(new S.SystemCall({ name: "$clog2" })), []);
equal(L.validate(new S.SystemCall({ name: "" })), []);
for (const bad of ["clog2", "$", "$a b", "$é"]) {
  equal(L.validate(new S.SystemCall({ name: bad })), [`'${bad}' is not a system task or function name`]);
}
equal(L.validate(new S.SystemCall({ name: 1n })), ["SystemCall.name must be a str, got int"]);
console.log("ok");

## VLG-03 · Kinds and features exist by family and year: Verilog has none of SystemVerilog's own

In [ ]:
equal([Verilog2005.STANDARD.name(), SystemVerilog2017.STANDARD.name(), SystemVerilog2023.STANDARD.name(), V1995.name()],
  ["Verilog-2005", "SystemVerilog-2017", "SystemVerilog-2023", "Verilog-1995"]);
const pkg = new S.PackageDeclaration({ name: name("p") });
equal(Verilog2005.check(pkg), ["PackageDeclaration is not Verilog"]);
equal(SV2005.check(pkg), []);
const ansi = new S.AnsiPort({ direction: "input", name: name("a") });
equal(V1995.check(ansi), ["AnsiPort needs Verilog-2001"]);
equal(V2001.check(ansi), []);
equal(Verilog2005.check(new S.IntegerVectorType({ keyword: "logic" })), ["IntegerVectorType.keyword 'logic' is not Verilog"]);
equal(Verilog2005.check(new S.IntegerVectorType({ keyword: "reg", signing: "signed" })), []);
equal(V1995.check(new S.IntegerVectorType({ keyword: "reg", signing: "signed" })),
  ["IntegerVectorType.signing 'signed' needs Verilog-2001"]);
const unique0 = new S.IfStatement({ qualifier: "unique0", condition: ref("a"), consequence: new S.NullStatement() });
equal(SV2005.check(unique0), ["IfStatement.qualifier 'unique0' needs SystemVerilog-2009"]);
equal(SV2009.check(unique0), []);
const final = new S.ImmediateAssertion({ keyword: "assert", deferral: "final", expression: ref("a") });
equal(SV2009.check(final), ["ImmediateAssertion.deferral 'final' needs SystemVerilog-2012"]);
equal(SV2012.check(final), []);
const star = new S.EventControl();
equal(V1995.check(star), ["EventControl * needs Verilog-2001"]);
equal(V2001.check(star), []);
equal(Verilog2005.check(new S.DisableStatement()), ["DisableStatement fork is not Verilog"]);
equal(Verilog2005.check(new S.DisableStatement({ target: ref("b") })), []);
const declared = new S.ForStatement({ initializers: [new S.VariableDeclaration({
  type: new S.IntegerAtomType({ keyword: "integer" }),
  declarators: [new S.VariableDeclarator({ name: name("i"), value: number() })] })], body: new S.NullStatement() });
equal(Verilog2005.check(declared), ["ForStatement.initializers declarations is not Verilog"]);
const assign = (target: string) => new S.AssignmentExpression({ target: ref(target), operator: "=", value: number() });
const two = new S.ForStatement({ initializers: [assign("i"), assign("j")], body: new S.NullStatement() });
equal(Verilog2005.check(two), ["ForStatement.initializers declarations is not Verilog"]);
equal(SV2005.check(two), []);
equal(Verilog2005.check(new S.ForStatement({ initializers: [assign("i")], body: new S.NullStatement() })), []);
equal(Verilog2005.check(module([], { labeled: true })), ["ModuleDeclaration.labeled is not Verilog"]);
equal(V1995.check(new S.BinaryExpression({ left: number(), operator: "**", right: number() })),
  ["BinaryExpression.operator '**' needs Verilog-2001"]);
equal(SV2005.check(new S.BinaryExpression({ left: number(), operator: "<->", right: number() })),
  ["BinaryExpression.operator '<->' needs SystemVerilog-2009"]);
console.log("ok");

## VLG-04 · A standard checks a whole tree, by path

In [ ]:
const tree = unit(module([new S.AlwaysConstruct({ keyword: "always_ff", body: new S.NullStatement() }),
  new S.ContinuousAssign({ assignments: [new S.AssignmentExpression({ target: ref("a"), operator: "+=", value: number() })] })]));
equal(Verilog2005.check(tree), ["items[0].items[0]: AlwaysConstruct.keyword 'always_ff' is not Verilog",
  "items[0].items[1].assignments[0]: AssignmentExpression.operator '+=' is not Verilog"]);
equal(SystemVerilog2023.check(tree), []);
console.log("ok");